In [0]:
# CONEXÃO E LEITURA DOS DADOS BRUTOS EM TXT
# 1. Credenciais 
storage_account_name = "datalakeivangui" 
storage_account_key = "uptABhZb3+kqJP0ac8jkdKnyKsjJkmKEfE9XjK0poagNWb85gzkwQYl7T+TXByh8ZYpPFa9dgPyJ+ASt0G1gQA=="         
container_name = "bronze-dados-brutos"

# 2. Caminhos para as duas pastas específicas
path_dueof = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/dueof/*.txt"
path_liquidacao = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/liquidacao/*.txt"

# 3. Leitura dos Empenhos (dueof)
df_dueof_bruto = spark.read \
    .option(f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net", storage_account_key) \
    .text(path_dueof)

# 4. Leitura das Liquidações
df_liq_bruto = spark.read \
    .option(f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net", storage_account_key) \
    .text(path_liquidacao)

# Testando se os dois funcionam
print("Amostra de Empenhos (dueof):")
display(df_dueof_bruto.limit(5))

print("Amostra de Liquidações:")
display(df_liq_bruto.limit(5))

Amostra de Empenhos (dueof):


value
03200428501130014900000002702200411963602 000000018267100104961800010900005 03MATERNIDADE DR ADALBERTO PEREIRA DA SILVA 00000000000000000000004270220043350430100000000000001 00000000000000000000000000000000000000000000000000000000000000000000000000099
03200428501870004100000000204200422716440 000000226765200104961800010900005 03MATERNIDADE DR ADALBERTO PEREIRA DA SILVA 00000000000000000000004020420043340690100000000000002 00000000000000000000000000000000000000000000000000000000000000000000000000099
03200428501130010300000002002200421743703 000000982507520074069600019200007 03PMH PRODUTOS MEDICOS HOSPITALARES LTDA 00000000000000000000004200220043390302800000000401122 00000000000000000000000000000000000000000000000000000000000000000000000000001
03200428501130049100000000204200423951931 000000005476450074069600019200007 03PMH PRODUTOS MEDICOS HOSPITALARES LTDA 00000000000000000000004020420043390302800000000401122 00000000000000000000000000000000000000000000000000000000000000000000000000007
03200428501130046200000000204200423952024 000000023056080074069600019200007 03PMH PRODUTOS MEDICOS HOSPITALARES LTDA 00000000000000000000004020420043390302800000000401122 00000000000000000000000000000000000000000000000000000000000000000000000000007


Amostra de Liquidações:


value
D201028500020000100000000000018200290420100929042010201000010005477 PAGAMENTO DE DI�RIAS 3011000229042010
D201028500020000100000000000013000261020100926102010201000010008770 PAGAMENTO DE DI�RIA NO PERCURSO S�O LU�S DE MONTES BELOS - C�RREGO DO OURO - S�O LU�S DE MONTES BELOS 3011000226102010
D201028500020000100000000000002600111120100911112010201000010012119 PAGAMENTO DE DI�RIAS NO PERCURSO S�O LUIS DE MONTES BELOS-GOI�S-S�O LUIS DE MONTES BELOS 3011000211112010
D201028500020000100000000000008000290920100929092010DR 201000010000881PAGAMENTO DE DI�RIAS NO PERCURSO RIO VERDE - GOI�NIA - RIO VERDE 3011000229092010
D201028500020000100000000000004000290920100929092010DR 201000010003426PAGAMENTO DE DI�RIAS NO PERCURSO RIO VERDE - GOI�NIA - RIO VERDE 3011000229092010


In [0]:
# --- OTIMIZAÇÃO: MEMORY CACHE (SEM GRAVAÇÃO EM DISCO) ---

print("Carregando e fixando dados na memória do cluster...")

# 1. O comando .cache() avisa ao Spark: "Leia o TXT uma vez e guarde na RAM"
df_dueof_pronto = df_dueof_bruto.cache()
df_liq_pronto = df_liq_bruto.cache()

# 2. O comando .count() é uma "Ação" que força o Spark a ler os dados AGORA
# Sem isso, o cache só aconteceria na próxima vez que você desse um display
print(f"DUEOF carregado: {df_dueof_pronto.count()} linhas.")
print(f"Liquidações carregadas: {df_liq_pronto.count()} linhas.")

# 3. Mantendo a compatibilidade com o resto do seu código
# df_dueof = df_dueof_pronto
# df_liq = df_liq_pronto

print("Sucesso! Os dados estão presos na memória e prontos para o fatiamento.")

Carregando e fixando dados na memória do cluster...
DUEOF carregado: 605800 linhas.
Liquidações carregadas: 891394 linhas.
Sucesso! Os dados estão presos na memória e prontos para o fatiamento.


In [0]:
from pyspark.sql.functions import col, trim, lit, concat, sum, when, substring, to_date, lpad, min
from pyspark.sql.types import DecimalType
from pyspark.sql import Window

In [0]:
#FATIA A TRIPA E CRIA UM DATAFRAME COM TODOS DADOS DISPONIVEIS
df_dueof_estruturado = df_dueof_pronto.select(
    col("value").substr(1, 2).alias("TIPO_DUEOF"),
    col("value").substr(3, 23).alias("CHAVE"),
    col("value").substr(3, 16).alias("NUMEMPENHO"),
    col("value").substr(26, 8).alias("DATA"),
    col("value").substr(34, 15).alias("NUMPROCESSO"),
    (col("value").substr(49, 14).cast(DecimalType(18, 2)) / 100).alias("VALOR"),
    col("value").substr(63, 14).alias("NUMR_CPF_CNPJ"),
    col("value").substr(77, 5).alias("FORMALIDADE_FINALIDADE"),
    col("value").substr(82, 11).alias("CONTA_BANCO_DEBITO"),
    col("value").substr(93, 11).alias("CONTA_BANCO_CREDITO"),
    col("value").substr(104, 2).alias("GRUPO_DESPESA"),
    col("value").substr(106, 60).alias("NOME_CREDOR"),
    col("value").substr(166, 9).alias("CODIGO_BANCO_DEBITO"),
    col("value").substr(175, 9).alias("CODIGO_BANCO_CREDITO"),
    col("value").substr(184, 3).alias("PRAZO_APLICACAO"),
    col("value").substr(187, 2).alias("STATUS_DOCUMENTO"),
    col("value").substr(189, 8).alias("DATA_DO_STATUS"),
    col("value").substr(197, 8).alias("NATUREZA"),
    col("value").substr(205, 13).alias("CODIGO_DO_PATRIMONIO"),
    col("value").substr(218, 1).alias("TIPO_EMPENHO"),
    col("value").substr(219, 10).alias("NUMERO_DECRETO"),
    col("value").substr(229, 8).alias("DATA_LEI"),
    col("value").substr(237, 7).alias("NUMERO_LEI"),
    col("value").substr(244, 2).alias("FUNCAO"),
    col("value").substr(246, 3).alias("SUB_FUNCAO"),
    col("value").substr(249, 4).alias("PROGRAMA"),
    col("value").substr(253, 4).alias("ACAO"),
    col("value").substr(257, 8).alias("FONTE"),
    col("value").substr(265, 12).alias("RECEITA_DEBITO"),
    col("value").substr(277, 12).alias("RECEITA_CREDITO"),
    col("value").substr(289, 13).alias("NUMERO_PDF"),
    col("value").substr(302, 2).alias("MODALIDADE_APLICACAO"),
    col("value").substr(304, 11).alias("FILLER")
)
# Visualização do resultado final
display(df_dueof_estruturado.limit(13))

TIPO_DUEOF,CHAVE,NUMEMPENHO,DATA,NUMPROCESSO,VALOR,NUMR_CPF_CNPJ,FORMALIDADE_FINALIDADE,CONTA_BANCO_DEBITO,CONTA_BANCO_CREDITO,GRUPO_DESPESA,NOME_CREDOR,CODIGO_BANCO_DEBITO,CODIGO_BANCO_CREDITO,PRAZO_APLICACAO,STATUS_DOCUMENTO,DATA_DO_STATUS,NATUREZA,CODIGO_DO_PATRIMONIO,TIPO_EMPENHO,NUMERO_DECRETO,DATA_LEI,NUMERO_LEI,FUNCAO,SUB_FUNCAO,PROGRAMA,ACAO,FONTE,RECEITA_DEBITO,RECEITA_CREDITO,NUMERO_PDF,MODALIDADE_APLICACAO,FILLER
03,20042850113001490000000,2004285011300149,27022004,11963602,18267.100000,01049618000109,00005,,,03,MATERNIDADE DR ADALBERTO PEREIRA DA SILVA,000000000,000000000,000,04,27022004,33504301,0000000000000,1,,00000000,0000000,00,000,0000,0000,00000000,000000000000,000000000000,0000000000000,00,99
03,20042850187000410000000,2004285018700041,02042004,22716440,226765.200000,01049618000109,00005,,,03,MATERNIDADE DR ADALBERTO PEREIRA DA SILVA,000000000,000000000,000,04,02042004,33406901,0000000000000,2,,00000000,0000000,00,000,0000,0000,00000000,000000000000,000000000000,0000000000000,00,99
03,20042850113001030000000,2004285011300103,20022004,21743703,982507.520000,00740696000192,00007,,,03,PMH PRODUTOS MEDICOS HOSPITALARES LTDA,000000000,000000000,000,04,20022004,33903028,0000000040112,2,,00000000,0000000,00,000,0000,0000,00000000,000000000000,000000000000,0000000000000,00,01
03,20042850113004910000000,2004285011300491,02042004,23951931,5476.450000,00740696000192,00007,,,03,PMH PRODUTOS MEDICOS HOSPITALARES LTDA,000000000,000000000,000,04,02042004,33903028,0000000040112,2,,00000000,0000000,00,000,0000,0000,00000000,000000000000,000000000000,0000000000000,00,07
03,20042850113004620000000,2004285011300462,02042004,23952024,23056.080000,00740696000192,00007,,,03,PMH PRODUTOS MEDICOS HOSPITALARES LTDA,000000000,000000000,000,04,02042004,33903028,0000000040112,2,,00000000,0000000,00,000,0000,0000,00000000,000000000000,000000000000,0000000000000,00,07
03,20042850113004660000000,2004285011300466,02042004,23951893,72427.860000,00740696000192,00007,,,03,PMH PRODUTOS MEDICOS HOSPITALARES LTDA,000000000,000000000,000,04,02042004,33903028,0000000040112,2,,00000000,0000000,00,000,0000,0000,00000000,000000000000,000000000000,0000000000000,00,07
03,20042850242000100000000,2004285024200010,20042004,24411485,632.160000,00740696000192,00001,,,03,PMH PRODUTOS MEDICOS HOSPITALARES LTDA,000000000,000000000,000,04,20042004,33903099,0000000010199,1,,00000000,0000000,00,000,0000,0000,00000000,000000000000,000000000000,0000000000000,00,99
03,20042850218003370000000,2004285021800337,11052004,24249882,11184.000000,00740696000192,00001,,,03,PMH PRODUTOS MEDICOS HOSPITALARES LTDA,000000000,000000000,000,04,11052004,33903028,0000000040112,1,,00000000,0000000,00,000,0000,0000,00000000,000000000000,000000000000,0000000000000,00,07
03,20042850020000360000000,2004285002000036,21052004,22840931,2601.960000,00740696000192,00001,,,04,PMH PRODUTOS MEDICOS HOSPITALARES LTDA,000000000,000000000,000,04,21052004,44905207,0000000010603,1,,00000000,0000000,00,000,0000,0000,00000000,000000000000,000000000000,0000000000000,00,07
03,20042850113017940000000,2004285011301794,01072004,24561487,8490.720000,00740696000192,00007,,,03,PMH PRODUTOS MEDICOS HOSPITALARES LTDA,000000000,000000000,000,04,01072004,33903028,0000000040112,2,,00000000,0000000,00,000,0000,0000,00000000,000000000000,000000000000,0000000000000,00,07


In [0]:
#FATIA A TRIPA E CRIA DOIS DATAFRAMES - TXT LIQ TEM 2 LAYOUTS
df_raw_D = df_liq_pronto.filter(col("value").substr(1, 1) == "D")
df_raw_M = df_liq_pronto.filter(col("value").substr(1, 1) == "M")

# ESTRUTURAÇÃO DO REGISTRO 'D' 
df_liq_D = df_raw_D.select(
    col("value").substr(1, 1).alias("TIPO_DUEOF"), 
    col("value").substr(2, 16).alias("NUMEMPENHO"),
    (col("value").substr(18, 17).cast(DecimalType(19, 2)) / 100).alias("VALOR"),
    col("value").substr(35, 8).alias("DATA"),
    col("value").substr(43, 2).alias("TIPODOCUMENTO"),
    col("value").substr(45, 8).alias("DATADOCUMENTO"),
    col("value").substr(53, 20).alias("NUMERODOCUMENTO"),
    trim(col("value").substr(73, 120)).alias("DESCRICAO"),
    col("value").substr(193, 9).alias("NUMEROSERIE"),
    col("value").substr(202, 5).alias("MODELONOTA"),
    col("value").substr(207, 8).alias("DATAVALIDADENOTA"),
    col("value").substr(215, 8).alias("DATA DE REFENCIA"),
    col("value").substr(223, 20).alias("NUMERO_FILA"),
    trim(col("value").substr(243, 150)).alias("DESC_FILA")
)

# ESTRUTURAÇÃO DO REGISTRO 'M'
df_liq_M = df_raw_M.select(
    col("value").substr(1, 1).alias("TIPO_DUEOF"),
    col("value").substr(2, 16).alias("NUMEMPENHO"),
    col("value").substr(18, 20).alias("NUMERODOCUMENTO"),
    col("value").substr(38, 4).alias("SEQMOVLIQ"),
    col("value").substr(42, 1).alias("TIPOMOVIMENTO"),
    (col("value").substr(43, 17).cast(DecimalType(19, 2)) / 100).alias("VALOR"),
    col("value").substr(60, 8).alias("DATA"),
    col("value").substr(68, 19).alias("OPREFERENCIA"),
    col("value").substr(87, 22).alias("MOVOPREFERENCIA"),
    col("value").substr(109, 1).alias("MOVOPTIPO"),
    col("value").substr(110, 105).alias("BRANCOS")
)

# Visualização para conferência
print("Amostra de Registros Tipo D:")
display(df_liq_D.limit(5))


print("Amostra de Registros Tipo M:")
display(df_liq_M.limit(5))

Amostra de Registros Tipo D:


TIPO_DUEOF,NUMEMPENHO,VALOR,DATA,TIPODOCUMENTO,DATADOCUMENTO,NUMERODOCUMENTO,DESCRICAO,NUMEROSERIE,MODELONOTA,DATAVALIDADENOTA,DATA DE REFENCIA,NUMERO_FILA,DESC_FILA
D,2010285000200001,182.000000,29042010,09,29042010,201000010005477,PAGAMENTO DE DI�RIAS,,,30110002,29042010,,
D,2010285000200001,130.000000,26102010,09,26102010,201000010008770,PAGAMENTO DE DI�RIA NO PERCURSO S�O LU�S DE MONTES BELOS - C�RREGO DO OURO - S�O LU�S DE MONTES BELOS,,30110,00226102,010,,
D,2010285000200001,26.000000,11112010,09,11112010,201000010012119,PAGAMENTO DE DI�RIAS NO PERCURSO S�O LUIS DE MONTES BELOS-GOI�S-S�O LUIS DE MONTES BELOS,,30,11000211,112010,,
D,2010285000200001,80.000000,29092010,09,29092010,DR 201000010000881,PAGAMENTO DE DI�RIAS NO PERCURSO RIO VERDE - GOI�NIA - RIO VERDE,,3,01100022,9092010,,
D,2010285000200001,40.000000,29092010,09,29092010,DR 201000010003426,PAGAMENTO DE DI�RIAS NO PERCURSO RIO VERDE - GOI�NIA - RIO VERDE,,3,01100022,9092010,,


Amostra de Registros Tipo M:


TIPO_DUEOF,NUMEMPENHO,NUMERODOCUMENTO,SEQMOVLIQ,TIPOMOVIMENTO,VALOR,DATA,OPREFERENCIA,MOVOPREFERENCIA,MOVOPTIPO,BRANCOS
M,2010285000200001,201000010005477,0001,7,182.000000,27122016,0000000000000000000,0000000000000000000000,0,
M,2010285000200001,201000010008770,0001,1,130.000000,12112010,2010285000200001269,0000000000000000000000,0,
M,2010285000200001,201000010012119,0001,7,26.000000,11112010,0000000000000000000,0000000000000000000000,0,
M,2010285000200001,DR 201000010000881,0001,7,80.000000,29092010,0000000000000000000,0000000000000000000000,0,
M,2010285000200001,DR 201000010003426,0001,1,40.000000,14102010,2010285000200001213,0000000000000000000000,0,


In [0]:
#AJUSTE DOS DATAFRAMES PARA MESMO LAYOUT E JUNÇÃO (DUEOF, LIQ_D, LIQ_M)

# Lista de tipos para o DUEOF
tipos_dueof_selecionados = [
    "01", "02", "03", "04", "05","06", "07", "08", "12", "13", "25", 
    "26", "27", "30", "31", "32", "33", "34", "35", "36", "37", "38", "46", "45"
]

# 1. Preparação do DUEOF (Filtrando os tipos específicos)
df_dueof_ready = df_dueof_estruturado.select(
    col("TIPO_DUEOF"),
    col("NUMEMPENHO"),
    col("VALOR"),
    col("DATA"),
    col("NUMPROCESSO")
).filter(col("TIPO_DUEOF").isin(tipos_dueof_selecionados))

# 2. Preparação do Liq_D (Todas as instâncias)
df_liq_D_ready = df_liq_D.select(
    col("TIPO_DUEOF"), 
    col("NUMEMPENHO"),
    col("VALOR"),
    col("DATA"),
    lit(None).alias("NUMPROCESSO")
)

# 3. Liq_M: Filtrando movimentos 6 e 7 e concatenando TIPO_DUEOF e TIPOMOVIMENTO
df_liq_M_ready = df_liq_M.select(
    concat(col("TIPO_DUEOF"), col("TIPOMOVIMENTO")).alias("TIPO_DUEOF"),
    col("NUMEMPENHO"),
    col("VALOR"),
    col("DATA"),
    lit(None).alias("NUMPROCESSO")
).filter(col("TIPOMOVIMENTO").isin("6", "7"))

# 4. CONSOLIDAÇÃO FINAL (Union das 3 fontes)
df_final_linhas = df_dueof_ready \
    .union(df_liq_D_ready) \
    .union(df_liq_M_ready)

# 5. --- ETAPA DE ETL: CORREÇÃO DE TIPOS ---
df_final_linhas = df_final_linhas.withColumn(
    # lpad garante que datas como 02012026 não percam o '0' à esquerda se forem tratadas como número
    "DATA", to_date(lpad(col("DATA").cast("string"), 8, '0'), "ddMMyyyy")
).withColumn(
    # DecimalType(18, 2) resolve o problema das casas decimais "pulando" no Excel
    "VALOR", col("VALOR").cast(DecimalType(18, 2))
)

# 6. Ordenação Lógica
df_final_linhas = df_final_linhas.orderBy("NUMEMPENHO", "DATA")

# Exibição do resultado
display(df_final_linhas.limit(5))

TIPO_DUEOF,NUMEMPENHO,VALOR,DATA,NUMPROCESSO
32,2003285000100000,8000.00,2003-01-13,
03,2003285000100001,23979.89,2003-12-29,23060450
D,2003285000100001,23979.89,2004-01-26,null
05,2003285000100001,23979.89,2004-03-18,23060450
02,2003285000100010,5000.00,2003-07-08,22953388


In [0]:
#DATAFRAME DOTACAO AGREGADO (df_dotacao)
dotacao_subtrai = ["02", "13", "33", "36", "38", "45"]
dotacao_soma = ["01", "12", "25", "34", "35", "37", "32", "46"]


df_dotacao = df_final_linhas \
    .withColumn("NUMDOTACAO", substring(col("NUMEMPENHO"), 1, 11)) \
    .groupBy("NUMDOTACAO") \
    .agg(
        
        sum(when(col("TIPO_DUEOF") == "32", col("VALOR")).otherwise(0)).alias("DOTACAO_INICIAL"),
        (
            #sum(when(col("TIPO_DUEOF").isin(array(*dotacao_soma))
            sum(when(col("TIPO_DUEOF").isin(dotacao_soma), col("VALOR")).otherwise(0)) - 
            #sum(when(col("TIPO_DUEOF").isin(array(*dotacao_subtrai))
            sum(when(col("TIPO_DUEOF").isin(dotacao_subtrai), col("VALOR")).otherwise(0))
        ).alias("DOTACAO_ATUALIZADA"),
        (
        sum(when(col("TIPO_DUEOF") == "03", col("VALOR")).otherwise(0)) + 
        sum(when(col("TIPO_DUEOF") == "31", col("VALOR")).otherwise(0)) +
        sum(when(col("TIPO_DUEOF") == "07", col("VALOR")).otherwise(0)) - 
        sum(when(col("TIPO_DUEOF") == "04", col("VALOR")).otherwise(0)) -
        sum(when(col("TIPO_DUEOF") == "30", col("VALOR")).otherwise(0))
        ).alias("DOTACAO_EMPENHADA"), #todos empenhos da dotação
        ((
            sum(when(col("TIPO_DUEOF").isin(dotacao_soma), col("VALOR")).otherwise(0)) - 
            sum(when(col("TIPO_DUEOF").isin(dotacao_subtrai), col("VALOR")).otherwise(0))
        ) -
        (
        sum(when(col("TIPO_DUEOF") == "03", col("VALOR")).otherwise(0)) + 
        sum(when(col("TIPO_DUEOF") == "31", col("VALOR")).otherwise(0)) +
        sum(when(col("TIPO_DUEOF") == "07", col("VALOR")).otherwise(0)) - 
        sum(when(col("TIPO_DUEOF") == "04", col("VALOR")).otherwise(0)) -
        sum(when(col("TIPO_DUEOF") == "30", col("VALOR")).otherwise(0))
        )).alias("DOTACAO_A_EMPENHAR"),
        min(when(col("TIPO_DUEOF") == "32", col("DATA"))).alias("DATA")   
        ) 

display(df_dotacao.limit(5))

NUMDOTACAO,DOTACAO_INICIAL,DOTACAO_ATUALIZADA,DOTACAO_EMPENHADA,DOTACAO_A_EMPENHAR,DATA
20042850040,1000.00,1000.00,0.00,1000.00,2004-01-16
20252850229,0.00,8268520.38,8268520.38,0.00,2025-04-09
20252850261,0.00,6000000.00,6000000.00,0.00,2025-01-29
20242850211,0.00,511544.22,239747.18,271797.04,2024-01-09
20242850052,10000.00,10000.00,0.00,10000.00,2024-01-09


In [0]:
# 1. Primeiro, criamos a agregação por NUMEMPENHO (os valores específicos do empenho)
df_empenho_agregado = df_final_linhas.groupBy("NUMEMPENHO").agg(
    (
        sum(when(col("TIPO_DUEOF") == "05", col("VALOR")).otherwise(0)) + 
        sum(when(col("TIPO_DUEOF") == "27", col("VALOR")).otherwise(0)) +
        sum(when(col("TIPO_DUEOF") == "08", col("VALOR")).otherwise(0)) - 
        sum(when(col("TIPO_DUEOF") == "26", col("VALOR")).otherwise(0)) -
        sum(when(col("TIPO_DUEOF") == "06", col("VALOR")).otherwise(0)) 
    ).alias("LIQUIDACAO_PAGO"), #saldo de ordem de pagamento
   (
        sum(when(col("TIPO_DUEOF") == "D", col("VALOR")).otherwise(0)) + 
        sum(when(col("TIPO_DUEOF") == "M6", col("VALOR")).otherwise(0)) - 
        sum(when(col("TIPO_DUEOF") == "M7", col("VALOR")).otherwise(0))
    ).alias("LIQUIDADO"), #saldo de liquidação
    (
        sum(when(col("TIPO_DUEOF") == "03", col("VALOR")).otherwise(0))
    ).alias("EMPENHO_INICIAL"), #primeiro empenho
    (
        sum(when(col("TIPO_DUEOF") == "03", col("VALOR")).otherwise(0)) + 
        sum(when(col("TIPO_DUEOF") == "31", col("VALOR")).otherwise(0)) +
        sum(when(col("TIPO_DUEOF") == "07", col("VALOR")).otherwise(0)) - 
        sum(when(col("TIPO_DUEOF") == "04", col("VALOR")).otherwise(0)) -
        sum(when(col("TIPO_DUEOF") == "30", col("VALOR")).otherwise(0))
    ).alias("EMPENHO_ATUALIZADO"), #empenho atualizado
    ((
        sum(when(col("TIPO_DUEOF") == "D", col("VALOR")).otherwise(0)) + 
        sum(when(col("TIPO_DUEOF") == "M6", col("VALOR")).otherwise(0)) - 
        sum(when(col("TIPO_DUEOF") == "M7", col("VALOR")).otherwise(0))
    ) - (
        sum(when(col("TIPO_DUEOF") == "05", col("VALOR")).otherwise(0)) + 
        sum(when(col("TIPO_DUEOF") == "27", col("VALOR")).otherwise(0)) +
        sum(when(col("TIPO_DUEOF") == "08", col("VALOR")).otherwise(0)) - 
        sum(when(col("TIPO_DUEOF") == "26", col("VALOR")).otherwise(0)) -
        sum(when(col("TIPO_DUEOF") == "06", col("VALOR")).otherwise(0))
    )).alias("LIQUIDACAO_A_PAGAR"), # LIQUIDADO - LIQUIDACAO_PAGO
    ((
        sum(when(col("TIPO_DUEOF") == "03", col("VALOR")).otherwise(0)) + 
        sum(when(col("TIPO_DUEOF") == "31", col("VALOR")).otherwise(0)) +
        sum(when(col("TIPO_DUEOF") == "07", col("VALOR")).otherwise(0)) - 
        sum(when(col("TIPO_DUEOF") == "04", col("VALOR")).otherwise(0)) -
        sum(when(col("TIPO_DUEOF") == "30", col("VALOR")).otherwise(0))
    )-(
        sum(when(col("TIPO_DUEOF") == "D", col("VALOR")).otherwise(0)) + 
        sum(when(col("TIPO_DUEOF") == "M6", col("VALOR")).otherwise(0)) - 
        sum(when(col("TIPO_DUEOF") == "M7", col("VALOR")).otherwise(0))
    )).alias("EMPENHO_A_LIQUIDAR") # EMPENHO_ATUALIZADO - LIQUIDADO


)

# 2. Criamos a chave de ligação (11 dígitos) no dataframe de empenhos
df_empenho_com_chave = df_empenho_agregado.withColumn("CHAVE_DOTACAO", substring(col("NUMEMPENHO"), 1, 11))

# 3. O JOIN Mágico: Cruzamos os empenhos com o df_dotacao que você já criou
df_relatorio_final = df_empenho_com_chave.join(
    df_dotacao, 
    df_empenho_com_chave.CHAVE_DOTACAO == df_dotacao.NUMDOTACAO, 
    "left"
)

# 4. Seleção final das colunas para ficar limpo
df_final = df_relatorio_final.select(
    "NUMEMPENHO",
    "NUMDOTACAO",
    "DOTACAO_INICIAL",  
    "DOTACAO_ATUALIZADA", 
    "DOTACAO_EMPENHADA",  
    "DOTACAO_A_EMPENHAR", 
    "EMPENHO_INICIAL", 
    "EMPENHO_ATUALIZADO", 
    "EMPENHO_A_LIQUIDAR", 
    "LIQUIDADO",
    "LIQUIDACAO_PAGO",
    "LIQUIDACAO_A_PAGAR"    
).orderBy("NUMEMPENHO")

# Visualização
display(df_final.limit(10))

NUMEMPENHO,NUMDOTACAO,DOTACAO_INICIAL,DOTACAO_ATUALIZADA,DOTACAO_EMPENHADA,DOTACAO_A_EMPENHAR,EMPENHO_INICIAL,EMPENHO_ATUALIZADO,EMPENHO_A_LIQUIDAR,LIQUIDADO,LIQUIDACAO_PAGO,LIQUIDACAO_A_PAGAR
2003285000100000,20032850001,8000.00,27000.00,23979.89,3020.11,0.00,0.00,0.00,0.00,0.00,0.00
2003285000100001,20032850001,8000.00,27000.00,23979.89,3020.11,23979.89,23979.89,0.00,23979.89,23979.89,0.00
2003285000100010,20032850001,8000.00,27000.00,23979.89,3020.11,0.00,0.00,0.00,0.00,0.00,0.00
2003285000200000,20032850002,5000.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2003285000200010,20032850002,5000.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2003285000300000,20032850003,500000.00,500000.00,0.00,500000.00,0.00,0.00,0.00,0.00,0.00,0.00
2003285000400000,20032850004,8000.00,19000.00,147.25,18852.75,0.00,0.00,0.00,0.00,0.00,0.00
2003285000400001,20032850004,8000.00,19000.00,147.25,18852.75,138.18,0.00,0.00,0.00,0.00,0.00
2003285000400002,20032850004,8000.00,19000.00,147.25,18852.75,147.52,147.25,0.00,147.25,147.25,0.00
2003285000400010,20032850004,8000.00,19000.00,147.25,18852.75,0.00,0.00,0.00,0.00,0.00,0.00


In [0]:
# EXPORTAÇÃO DAS TABELAS EM CSV

# 1. Configurações de Caminho e Credencial Global
container_silver = "silver-dados-processados"
base_path = f"wasbs://{container_silver}@{storage_account_name}.blob.core.windows.net/tabelas_csv"

# ISSO AQUI É A CHAVE: Define a permissão para TODO o notebook de uma vez
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net", 
    storage_account_key
)

# 2. DEFINA OS NOMES PERSONALIZADOS
export_map = {
    "df_final_linhas": "base_eventos_linhas",
    "df_dotacao": "dotacao_consolidada",
    "df_final": "painel_empenhos_consolidado"
}

print("Iniciando exportação com nomes fixos e credenciais globais...")

for df_origem, nome_personalizado in export_map.items():
    temp_folder = f"{base_path}/{nome_personalizado}_temp"
    final_file_path = f"{base_path}/{nome_personalizado}.csv"
    
    # Pega o DataFrame real pelo nome
    df_para_gravar = globals()[df_origem]
    
    print(f"Gerando arquivo: {nome_personalizado}.csv")
    
    # Etapa 1: Grava como pasta (agora a credencial já está no sistema)
    df_para_gravar.coalesce(1).write.format("csv") \
      .mode("overwrite") \
      .option("header", "true") \
      .option("delimiter", ";") \
      .save(temp_folder)
    
    # Etapa 2: Localiza o arquivo e renomeia
    # Agora o dbutils terá acesso porque configuramos o spark.conf.set acima
    files = dbutils.fs.ls(temp_folder)
    csv_temp_file = [f.path for f in files if f.path.endswith(".csv")][0]
    
    dbutils.fs.cp(csv_temp_file, final_file_path)
    dbutils.fs.rm(temp_folder, recurse=True)

print("--- Sucesso! Arquivos nomeados corretamente no Azure. ---")

Iniciando exportação com nomes fixos e credenciais globais...
Gerando arquivo: base_eventos_linhas.csv
Gerando arquivo: dotacao_consolidada.csv
Gerando arquivo: painel_empenhos_consolidado.csv
--- Sucesso! Arquivos nomeados corretamente no Azure. ---
